In [2]:
import sys
print(sys.executable)

c:\Users\sec\AppData\Local\Programs\Python\Python312\python.exe


In [3]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
print("ok")

ok


In [9]:
import requests

url = "https://apis.data.go.kr/B550373/kdhcSolarPowerProd/solarPowerProd"

params = {
    "serviceKey": "0Fu9Z5Pqs1ZE7dRq092g6CvbEBIgLeu6AU9/tJy0ETZN4CXiLIKZQB53zT0C+2r3zMbPFJ14t8VyF3XV0RlL3Q==",
    "pageNo": "1",
    "numOfRows": "10",
    "startDate": "20240101",
    "endDate": "20240131"
}

response = requests.get(url, params=params)

print("status:", response.status_code)
print("final url:", response.url)
print(response.text[:1000])

status: 200
final url: https://apis.data.go.kr/B550373/kdhcSolarPowerProd/solarPowerProd?serviceKey=0Fu9Z5Pqs1ZE7dRq092g6CvbEBIgLeu6AU9%2FtJy0ETZN4CXiLIKZQB53zT0C%2B2r3zMbPFJ14t8VyF3XV0RlL3Q%3D%3D&pageNo=1&numOfRows=10&startDate=20240101&endDate=20240131
<?xml version="1.0" encoding="UTF-8" standalone="yes"?><response><header><resultCode>00</resultCode><resultMsg>NORMAL SERVICE.</resultMsg></header><body><items><item><accumGen>423155</accumGen><genName>판교(가압장)</genName><rnum>10</rnum><workHr>14</workHr><ymd>20240131</ymd></item><item><accumGen>423165</accumGen><genName>판교(가압장)</genName><rnum>9</rnum><workHr>15</workHr><ymd>20240131</ymd></item><item><accumGen>423171</accumGen><genName>판교(가압장)</genName><rnum>8</rnum><workHr>16</workHr><ymd>20240131</ymd></item><item><accumGen>423171</accumGen><genName>판교(가압장)</genName><rnum>7</rnum><workHr>17</workHr><ymd>20240131</ymd></item><item><accumGen>423171</accumGen><genName>판교(가압장)</genName><rnum>6</rnum><workHr>18</workHr><ymd>20240131</ymd><

In [10]:
import os
import time
import requests

service_key = "0Fu9Z5Pqs1ZE7dRq092g6CvbEBIgLeu6AU9/tJy0ETZN4CXiLIKZQB53zT0C+2r3zMbPFJ14t8VyF3XV0RlL3Q=="
base_url = "https://apis.data.go.kr/B550373/kdhcSolarPowerProd/solarPowerProd"

save_dir = "xml_data"
os.makedirs(save_dir, exist_ok=True)

page_no = 1
num_of_rows = 100
max_pages = 10   # 100개씩 10페이지 = 최대 1000 row 가정

for page_no in range(1, max_pages + 1):
    params = {
        "serviceKey": service_key,
        "pageNo": str(page_no),
        "numOfRows": str(num_of_rows),
        "startDate": "20240101",
        "endDate": "20240131"
    }

    response = requests.get(base_url, params=params, timeout=30)

    print(f"page {page_no} status:", response.status_code)
    print("url:", response.url)

    if response.status_code != 200:
        print(f"{page_no}페이지 요청 실패")
        break

    xml_text = response.text

    # 에러 응답 방지용 간단 체크
    if "Unexpected errors" in xml_text:
        print(f"{page_no}페이지에서 서버 에러 발생")
        break

    file_path = os.path.join(save_dir, f"solarPowerProd_page_{page_no}.xml")
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(xml_text)

    print(f"저장 완료: {file_path}")

    time.sleep(0.3)  # 너무 빠른 연속 호출 방지

page 1 status: 200
url: https://apis.data.go.kr/B550373/kdhcSolarPowerProd/solarPowerProd?serviceKey=0Fu9Z5Pqs1ZE7dRq092g6CvbEBIgLeu6AU9%2FtJy0ETZN4CXiLIKZQB53zT0C%2B2r3zMbPFJ14t8VyF3XV0RlL3Q%3D%3D&pageNo=1&numOfRows=100&startDate=20240101&endDate=20240131
저장 완료: xml_data\solarPowerProd_page_1.xml
page 2 status: 200
url: https://apis.data.go.kr/B550373/kdhcSolarPowerProd/solarPowerProd?serviceKey=0Fu9Z5Pqs1ZE7dRq092g6CvbEBIgLeu6AU9%2FtJy0ETZN4CXiLIKZQB53zT0C%2B2r3zMbPFJ14t8VyF3XV0RlL3Q%3D%3D&pageNo=2&numOfRows=100&startDate=20240101&endDate=20240131
저장 완료: xml_data\solarPowerProd_page_2.xml
page 3 status: 200
url: https://apis.data.go.kr/B550373/kdhcSolarPowerProd/solarPowerProd?serviceKey=0Fu9Z5Pqs1ZE7dRq092g6CvbEBIgLeu6AU9%2FtJy0ETZN4CXiLIKZQB53zT0C%2B2r3zMbPFJ14t8VyF3XV0RlL3Q%3D%3D&pageNo=3&numOfRows=100&startDate=20240101&endDate=20240131
저장 완료: xml_data\solarPowerProd_page_3.xml
page 4 status: 200
url: https://apis.data.go.kr/B550373/kdhcSolarPowerProd/solarPowerProd?serviceKey=0F

In [11]:
import os
import glob
import xml.etree.ElementTree as ET

input_folder = "xml_data"          # xml 파일들이 들어있는 폴더
output_file = "solarPowerProd.xml"

# 최종 루트 생성
merged_root = ET.Element("response")
merged_body = ET.SubElement(merged_root, "body")
merged_items = ET.SubElement(merged_body, "items")

xml_files = sorted(glob.glob(os.path.join(input_folder, "*.xml")))

total_count = 0

for file_path in xml_files:
    tree = ET.parse(file_path)
    root = tree.getroot()

    items = root.findall(".//item")
    for item in items:
        merged_items.append(item)
        total_count += 1

# totalCount 같은 메타 정보 추가
total_count_tag = ET.SubElement(merged_body, "totalCount")
total_count_tag.text = str(total_count)

# 저장
merged_tree = ET.ElementTree(merged_root)
ET.indent(merged_tree, space="  ", level=0)  # Python 3.9+
merged_tree.write(output_file, encoding="utf-8", xml_declaration=True)

print(f"합쳐진 파일 저장 완료: {output_file}")
print(f"총 item 수: {total_count}")

합쳐진 파일 저장 완료: solarPowerProd.xml
총 item 수: 1000
